# 01 · Explore the data
Load `twcs.csv`, sanity-check brand volumes, sample raw Uber_Support inbound tweets to feel the mess.

Exploration only — the graded artifact is `src/`. Run from repo root.

In [1]:
import os, sys
os.environ['PYTHONUTF8'] = '1'
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
RAW = os.path.join('..', 'data', 'twitter_support', 'twcs.csv')
df = pd.read_csv(RAW, usecols=['tweet_id','author_id','inbound','text','in_response_to_tweet_id'],
                 dtype={'author_id':'string'})
print('total rows:', len(df))
df.head()

total rows: 2811774


,tweet_id,author_id,inbound,text,in_response_to_tweet_id
0,1,sprintcare,False,@115712 I understand. I would like to assist y...,3.0
1,2,115712,True,@sprintcare and how do you propose we do that,1.0
2,3,115712,True,@sprintcare I have sent several private messag...,4.0
3,4,sprintcare,False,@115712 Please send us a Private Message so th...,5.0
4,5,115712,True,@sprintcare I did.,6.0


## Brand volumes — where does Uber_Support rank?

In [2]:
brand_out = df[df['inbound'] == False]['author_id'].value_counts()
print('Top 10 brands by outbound volume:')
print(brand_out.head(10))
print('\nUber_Support outbound:', int(brand_out.get('Uber_Support', 0)))

Top 10 brands by outbound volume:
author_id
AmazonHelp         169840
AppleSupport       106860
Uber_Support        56270
SpotifyCares        43265
Delta               42253
Tesco               38573
AmericanAir         36764
TMobileHelp         34317
comcastcares        33031
British_Airways     29361
Name: count, dtype: int64[pyarrow]

Uber_Support outbound: 56270


In [3]:
# inbound tweets @-mentioning Uber_Support (customers writing in)
inbound_uber = df[(df['inbound'] == True) & (df['text'].str.contains('Uber_Support|115873|115877', case=False, na=False))]
print('inbound mentioning Uber:', len(inbound_uber))

inbound mentioning Uber: 69242


## Sample raw inbound tweets — typos, media links, mixed intents

In [4]:
for t in inbound_uber['text'].sample(15, random_state=1):
    print('-', t[:150])

- @115877 @148768 Why did I only receive half my order ref - #EAEF5 !? REFUND OR REPLACE NOW!
- @115873 @Uber_Support your drivers are a disgrace! Cancelling our job as we walked to the car because he picked up a higher fare instead! Leaving 3 la
- @Uber_Support @131434 https://t.co/PMQTJ6KAMy
- @115873 had 2 drivers call and cancel on me cause I'm not in there exact Location.... shady mofos
- @Uber_Support Been there.Done that.Your team is insisting that the fee is warranted.SO UNFAIR! I did not do anything to this driver's vehicle!
- @115873  and @115879 As someone who is considering being a driver in the future, is there going to be any support you might be able to hint at @115858
- @Uber_Support The ride isn’t in my recent rides but I was still charged on my card the full ride charge
- @Uber_Support Done. Can you confirm if the trip was cancelled, will the payment be returned automatically? Thanks
- @Uber_Support Have been trying since 11.37 in the morning. No credible response fro

**Observations:** billing/refund complaints dominate; safety/trip issues are higher-risk minority; many terse follow-ups ('check DM', 'already did that') carry no intent without the parent tweet — which motivates thread reconstruction in notebook 02.